In [1]:
import xarray as xr
import json
import yaml

In [2]:
adf_loc = "/glade/u/home/qingyuany/ADF/"
obs_loc = "/glade/campaign/cgd/amp/amwg/ADF_obs/"

In [3]:
with open(adf_loc + "lib/adf_variable_defaults.yaml", "r") as f:
    obs_info = yaml.safe_load(f)

In [6]:
obs_info['FSNTC']

{'category': 'TOA energy flux',
 'obs_file': 'CERES_EBAF_Ed4.1_2001-2020.nc',
 'obs_name': 'CERES_EBAF_Ed4.1',
 'obs_var_name': 'toa_sw_clr_t_mon',
 'pct_diff_contour_levels': [-100,
  -75,
  -50,
  -40,
  -30,
  -20,
  -10,
  -8,
  -6,
  -4,
  -2,
  0,
  2,
  4,
  6,
  8,
  10,
  20,
  30,
  40,
  50,
  75,
  100],
 'pct_diff_colormap': 'PuOr_r'}

In [7]:
obs_info["FLUTC"]

{'obs_file': 'CERES_EBAF_Ed4.1_2001-2020.nc',
 'obs_name': 'CERES_EBAF_Ed4.1',
 'obs_var_name': 'toa_lw_clr_t_mon'}

In [5]:
voi = ['RESTOM', 'FSNT', 'FLNT', 'SWCF', 'LWCF', 'PRECT', 'TGCLDLWP', 'FLUTC', 'FSNTC', 'TMQ', 'LHFLX', 'FSNTC', 'FLUTC']


In [9]:
camoutput = xr.open_dataset("/glade/work/qingyuany/cam7_tuning/cam7/ppe_time_average_nc/ppe1.nc")

In [10]:
camoutput

<xarray.Dataset> Size: 273MB
Dimensions:   (ppe_ind: 95, lat: 192, lon: 288)
Coordinates:
  * lat       (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon       (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
  * ppe_ind   (ppe_ind) int64 760B 1 2 3 4 5 6 7 8 9 ... 92 93 94 95 96 97 98 99
Data variables: (12/13)
    FSNTOA    (ppe_ind, lat, lon) float32 21MB ...
    FLUT      (ppe_ind, lat, lon) float32 21MB ...
    FSNT      (ppe_ind, lat, lon) float32 21MB ...
    FLNT      (ppe_ind, lat, lon) float32 21MB ...
    SWCF      (ppe_ind, lat, lon) float32 21MB ...
    LWCF      (ppe_ind, lat, lon) float32 21MB ...
    ...        ...
    TGCLDLWP  (ppe_ind, lat, lon) float32 21MB ...
    FSNTC     (ppe_ind, lat, lon) float32 21MB ...
    TMQ       (ppe_ind, lat, lon) float32 21MB ...
    LHFLX     (ppe_ind, lat, lon) float32 21MB ...
    RESTOM    (ppe_ind, lat, lon) float32 21MB ...
    RESTOA    (ppe_ind, lat, lon) float32 21MB ...

In [7]:
var_obs_dict = {}

for v in voi:
    t_where = obs_info[v]['obs_file']
    t_var_nm = obs_info[v]['obs_var_name']
    print("{:<10} {:<50} {:<10}".format(v, t_where, t_var_nm))
    var_obs_dict[v] = t_var_nm

RESTOM     CERES_EBAF_Ed4.1_2001-2020.nc                      toa_net_all_mon
FSNT       CERES_EBAF_Ed4.1_2001-2020.nc                      fsnt      
FLNT       CERES_EBAF_Ed4.1_2001-2020.nc                      toa_lw_all_mon
SWCF       CERES_EBAF_Ed4.1_2001-2020.nc                      toa_cre_sw_mon
LWCF       CERES_EBAF_Ed4.1_2001-2020.nc                      toa_cre_lw_mon
PRECT      ERAI_all_climo.nc                                  PRECT     
TGCLDLWP   TGCLDLWP_ERA5_monthly_climo_197901-202112.nc       TGCLDLWP  
FLUTC      CERES_EBAF_Ed4.1_2001-2020.nc                      toa_lw_clr_t_mon
FSNTC      CERES_EBAF_Ed4.1_2001-2020.nc                      toa_sw_clr_t_mon
TMQ        ERAI_all_climo.nc                                  PREH2O    
LHFLX      ERAI_all_climo.nc                                  LHFLX     


In [8]:
var_obs_dict

{'RESTOM': 'toa_net_all_mon',
 'FSNT': 'fsnt',
 'FLNT': 'toa_lw_all_mon',
 'SWCF': 'toa_cre_sw_mon',
 'LWCF': 'toa_cre_lw_mon',
 'PRECT': 'PRECT',
 'TGCLDLWP': 'TGCLDLWP',
 'FLUTC': 'toa_lw_clr_t_mon',
 'FSNTC': 'toa_sw_clr_t_mon',
 'TMQ': 'PREH2O',
 'LHFLX': 'LHFLX'}

In [9]:
obs = []
for v in voi:
    temp_v = xr.open_dataset(obs_loc + obs_info[v]["obs_file"])[obs_info[v]['obs_var_name']]

    temp_v = temp_v.interp(lat=camoutput.lat, lon=camoutput.lon, method="nearest")
    print(temp_v.shape)

    obs.append(temp_v.mean(dim = "time"))

(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)


In [10]:
obs_ds = xr.Dataset({da.name: da for da in obs})

In [14]:
obs_ds

<xarray.Dataset> Size: 2MB
Dimensions:           (lat: 192, lon: 288)
Coordinates:
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
Data variables:
    toa_net_all_mon   (lat, lon) float32 221kB nan nan nan nan ... nan nan nan
    fsnt              (lat, lon) float32 221kB nan nan nan nan ... nan nan nan
    toa_lw_all_mon    (lat, lon) float32 221kB nan nan nan nan ... nan nan nan
    toa_cre_sw_mon    (lat, lon) float32 221kB nan nan nan nan ... nan nan nan
    toa_cre_lw_mon    (lat, lon) float32 221kB nan nan nan nan ... nan nan nan
    PRECT             (lat, lon) float32 221kB 0.1423 0.1423 ... 0.6106 nan
    TGCLDLWP          (lat, lon) float32 221kB 3.899e-05 3.899e-05 ... 0.03124
    toa_lw_clr_t_mon  (lat, lon) float32 221kB nan nan nan nan ... nan nan nan
    toa_sw_clr_t_mon  (lat, lon) float32 221kB nan nan nan nan ... nan nan nan
    PREH2O            (lat, lon) float32 221kB 0.5262 0.5262 ... 5.276 nan
    LHFLX             (lat, lon) float32 221kB -0.1057 -0.1057 ... 3.233 nan

In [11]:
pr = xr.open_dataset(obs_loc + obs_info["PRECT"]["obs_file"])[obs_info["PRECT"]['obs_var_name']]
pr

<xarray.DataArray 'PRECT' (time: 12, lat: 121, lon: 240)> Size: 1MB
[348480 values with dtype=float32]
Coordinates:
  * time     (time) float32 48B 1.0 2.0 3.0 4.0 5.0 ... 8.0 9.0 10.0 11.0 12.0
  * lat      (lat) float32 484B -90.0 -88.5 -87.0 -85.5 ... 85.5 87.0 88.5 90.0
  * lon      (lon) float32 960B 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
Attributes:
    center:        European Center for Medium-Range Weather Forecasts (RSMC)
    info:          function clmMonLLT: contributed.ncl
    long_name:     Total precipitation
    time_op_ncl:   Climatology: 17 years
    units:         mm/day
    cell_methods:  time: mean

In [15]:
obs_ds["PRECT"] = obs_ds["PRECT"]/(1000 * 24 * 3600)

86400

In [16]:
#obs_ds.to_netcdf("/glade/work/qingyuany/cam7/obs/obs.nc")